# K-Nearest Neighbors: From First Principles

## Why Do We Need KNN?

Some predictions can be made by finding examples that resemble the new case. A home may have a similar price to nearby homes with similar size and location. A flower may belong to the same species as flowers with similar measurements. K-Nearest Neighbors (KNN) turns this intuition into a method: represent each example by its features, measure how far examples are from one another, and use the closest training examples to make a prediction.

Unlike Linear Regression, KNN does not learn a fixed set of coefficients. Unlike a Decision Tree, it does not build a sequence of splits. It mainly stores the training examples and delays most of the work until a prediction is requested. This is why KNN is called a **lazy, instance-based learner**.

<div style="background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>When features describe meaningful similarity, nearby training examples can provide useful evidence about a new example. KNN's predictions are only as sensible as the distance measure and features that define "nearby."</div>

This lesson develops that idea in order: define the two KNN tasks, understand distance, work through a prediction, and then see how the choices of $K$, feature scaling, and validation affect results.

## What Problem Does KNN Solve?

KNN can solve both classification and regression. The neighborhood is selected in the same way, but the target values are combined differently:

| Task | What neighbors contain | KNN prediction | Example |
|---|---|---|---|
| Classification | Class labels | The most common class among the neighbors | Predict a flower species |
| Regression | Numerical target values | The mean (or a distance-weighted mean) of neighbor targets | Estimate a home price |

The letter $K$ is the number of training examples used. With $K=5$, KNN finds the five nearest training examples and combines their targets. $K$ is a model setting, not the number of classes and not the number of features.

<div style="background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Important distinction</strong><br>KNN does not average feature values to create a new training example. It finds the neighbors in feature space, then combines their known target labels or values.</div>

## Distance-Based Learning

Each example is represented as a point in feature space. With two features, such as height and weight, a person is a point on a two-dimensional graph. With $d$ features, the point has $d$ coordinates. A distance function turns the difference between two feature vectors into one number: smaller distance means the examples are considered more similar by that metric.

For a new sample, KNN compares its feature vector with the stored training vectors. The nearest examples form its neighborhood. If nearby examples tend to share labels or have similar target values, their targets provide evidence for the new prediction.

This method assumes that the selected features and distance metric reflect meaningful similarity. Two records can be numerically close but unrelated in the real problem if irrelevant features dominate or the feature representation is poor. The metric is part of the model design, not a neutral measurement.

<div style="background-color:#fff5df; border-left:5px solid #d18a00; padding:10px 14px; margin:12px 0;"><strong>Check before trusting a neighborhood</strong><br>Ask whether a small distance in the chosen features should really mean that two cases have similar outcomes. If the answer is unclear, reconsider the features, their scaling, or the distance metric.</div>

## Euclidean Distance

Euclidean distance is the straight-line distance between two points. In two dimensions, it follows from the Pythagorean theorem. For $A(x_1,y_1)$ and $B(x_2,y_2)$:

$$
d(A,B)=\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}
$$

For example, the distance from $(1,2)$ to $(4,6)$ is $\sqrt{3^2+4^2}=5$. With $d$ features, the same idea extends to

$$
d(\mathbf{x},\mathbf{a})=\sqrt{\sum_{j=1}^{d}(x_j-a_j)^2}
$$

Here, $\mathbf{x}$ is the new example and $\mathbf{a}$ is one training example. The differences in each feature are squared, summed, and square-rooted. Squaring means a large difference on one feature can contribute much more than a small difference on another.

Euclidean distance is common for continuous numerical features, especially when directions and magnitudes are meaningful. It is sensitive to measurement scale and can be strongly affected by outliers, so inspect and scale features before using it.

## Manhattan Distance and Comparing Metrics

Manhattan distance adds the absolute change along each feature axis. In two dimensions, it is the distance traveled by moving along the grid rather than diagonally:

$$
d(A,B)=|x_2-x_1|+|y_2-y_1|
$$

For $d$ features:

$$
d(\mathbf{x},\mathbf{a})=\sum_{j=1}^{d}|x_j-a_j|
$$

The two common metrics behave differently:

| Metric | Formula idea | Intuition | Consider when |
|---|---|---|---|
| Euclidean ($L_2$) | Square differences, sum them, take the square root | Straight-line distance | Continuous measurements where diagonal closeness is meaningful |
| Manhattan ($L_1$) | Sum absolute feature differences | Grid or axis-by-axis distance | Accumulated per-feature differences are meaningful |

Both metrics can change which examples count as nearest neighbors, and both are affected by feature scaling. Neither is universally best. The appropriate choice depends on what the features represent and should be selected using validation rather than the test set.

## Choosing $K$

$K$ controls how wide a neighborhood influences the prediction. With a small $K$, only very local examples vote or contribute to the average. This can capture fine detail, but one noisy or mislabeled point can have a large effect. With a large $K$, the prediction averages evidence across a broader region. It is more stable but can blur genuine local structure.

| Choice of $K$ | Typical behavior | Common risk |
|---|---|---|
| Too small | Very local, flexible predictions | Overfits noise; high variance |
| Too large | Broad, smooth predictions | Underfits local patterns; high bias |

There is no universal best value. Try a range of plausible values with cross-validation using only the training data, and choose based on the metric that matches the task. Then evaluate the selected model once on held-out test data. For binary classification, an odd $K$ can reduce ties, but it does not guarantee no tie in every multiclass or weighted-voting setting.

Some KNN implementations can weight votes by distance so closer neighbors count more than farther neighbors. Uniform voting gives each of the $K$ neighbors equal weight. The weighting rule is another model choice and can also be compared during validation.

## Feature Scaling

Distance combines differences across all features. Suppose age ranges from 0 to 100 while income ranges from 0 to 200,000. A difference of 10 years contributes far less numerically than a difference of 10,000 dollars, so income may dominate the distance even if age is just as important to the task. KNN has no built-in understanding of units or importance.

Standardization is a common remedy. For a feature value $x$, it computes

$$
z=\frac{x-\mu}{\sigma}
$$

where $\mu$ and $\sigma$ are the mean and standard deviation estimated from training data. Min-max scaling is another option when a fixed range is useful. Scaling makes values more comparable numerically, but it does not decide which features are meaningful; irrelevant or redundant features can still distort distances.

<div style="background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Prevent data leakage</strong><br>Fit the scaler on training data only, then apply that fitted scaler to validation and test data. In cross-validation, put scaling and KNN in one pipeline so each fold learns its scaling statistics from that fold's training portion.</div>

For categorical variables, choose an encoding and distance interpretation that make sense. Arbitrary integer codes can falsely imply that category 3 is farther from category 1 than category 2 is. One-hot encoding avoids that particular ordering, but its effect on distance should still be considered.

## How KNN Makes a Prediction

For a new sample, the prediction process is:

1. Represent the new sample and all training samples using the same features and preprocessing.
2. Compute the distance from the new sample to each training sample.
3. Select the $K$ samples with the smallest distances.
4. Combine their known targets: vote for a class or average numerical values.

For uniform classification, the predicted class is the one with the most neighbor votes. If classes tie, the implementation needs a tie-breaking rule. With distance weighting, closer neighbors contribute more to the vote. For uniform regression, the prediction is the mean of the neighbor targets:

$$
\hat{y}=\frac{1}{K}\sum_{i=1}^{K}y_{(i)}
$$

Here, $y_{(i)}$ is the target value for the $i$th nearest neighbor. A distance-weighted regression prediction uses a weighted mean, with larger weights assigned to closer neighbors. Exact behavior for zero distances and ties depends on the implementation.

## Worked Example

Assume sweetness and weight have already been scaled, and a new fruit is at $(0,0)$ in that feature space. Four training fruits have the coordinates and distances below, using Euclidean distance:

| Fruit | Coordinates | Class | Distance to new fruit |
|---|---:|---|---:|
| 1 | $(1,0)$ | Apple | 1 |
| 2 | $(0,1)$ | Orange | 1 |
| 3 | $(-2,0)$ | Apple | 2 |
| 4 | $(0,-3)$ | Orange | 3 |

With $K=3$ and uniform voting, the neighborhood contains two apples and one orange, so KNN predicts **Apple**. The fourth fruit does not vote because it is outside the three nearest examples. If $K$ changed, the neighborhood and potentially the prediction could change.

For regression, suppose the three nearest homes have prices of $300{,}000$, $320{,}000$, and $310{,}000$. Uniform KNN predicts their mean:

$$
\hat{y}=\frac{300{,}000+320{,}000+310{,}000}{3}=310{,}000
$$

<div style="background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>What to notice</strong><br>KNN does not infer a global equation in this example. It uses the local training examples around the new point, so the selected features, scaling, distance metric, and $K$ jointly determine the neighborhood and prediction.</div>

## The Curse of Dimensionality

As the number of features grows, the volume of feature space grows rapidly. With a fixed amount of training data, examples become sparse, and the nearest and farthest distances can become less distinct. A neighborhood that seemed meaningfully local in two dimensions may be much less informative across hundreds of dimensions.

Irrelevant features make the problem worse because their differences add to the distance even when they tell us nothing about the target. Correlated features can also count nearly the same signal multiple times. This is why scaling alone is not sufficient: it balances units, but does not remove unhelpful dimensions.

Feature selection, domain-informed representations, or dimensionality reduction can help, but each must be fitted within the training process and evaluated carefully. KNN is often most effective when the feature space is moderate in size and the chosen features define a trustworthy notion of similarity.

## Computational Cost

KNN is called a lazy learner because fitting usually does not search for coefficients or split rules; it stores the training examples and any required preprocessing. The exact fit cost depends on whether the implementation copies or indexes the data, but storage is generally proportional to the training-set size.

With a brute-force search, predicting one sample requires computing distances to all $n$ training rows across $d$ features, which is approximately $O(nd)$ work. Finding the nearest $K$ can add selection or sorting cost; a full sort is typically $O(n\log n)$. For $m$ query samples, the distance-computation work is approximately $O(mnd)$. The model must also retain the training data, requiring roughly $O(nd)$ memory, in addition to preprocessing or index structures.

For small datasets this simplicity is convenient. For large datasets, prediction latency and memory can become a bottleneck. Specialized neighbor-search indexes can reduce work in some low-dimensional settings, but their benefit often weakens as dimensionality increases. Measure prediction time on a representative workload when speed matters.

## Advantages, Disadvantages, and a Practical Workflow

KNN is easy to explain, can model nonlinear decision boundaries, and naturally supports both classification and regression. It makes few assumptions about the overall shape of the relationship. Its trade-offs follow directly from its design:

| Strengths | Limitations |
|---|---|
| Simple, intuitive local predictions | Prediction can be slow because many stored examples are compared |
| Can represent nonlinear patterns | Must retain the training data, increasing memory use |
| Supports classification and regression | Sensitive to scaling, irrelevant features, and the distance metric |
| Few assumptions about a global functional form | Can struggle with sparse, high-dimensional data |

A sound workflow is to split the data first, then place preprocessing and KNN in a pipeline. Compare plausible distance metrics, values of $K$, and (if available) uniform versus distance weighting with cross-validation on the training data. Select settings using a task-appropriate validation metric, refit on the training data, and report final performance on the held-out test set only once.

For classification, accuracy can be misleading when classes are imbalanced; examine a confusion matrix and metrics such as precision, recall, and F1 for the classes that matter. ROC-AUC may be useful when probability-like scores and ranking quality are relevant. For regression, MAE is an interpretable average absolute error, RMSE penalizes large errors more strongly, and $R^2$ compares the fit with a mean-prediction baseline. Use metrics that reflect the actual cost of mistakes.

<div style="background-color:#fff5df; border-left:5px solid #d18a00; padding:10px 14px; margin:12px 0;"><strong>Rule of thumb</strong><br>Use KNN when meaningful similarity can be captured by a manageable set of features and prediction latency is acceptable. Be cautious when the data is large, the feature space is high-dimensional, or distance does not correspond to outcome similarity.</div>